# 12 — SHAP explanations and LLM handoff

This notebook explains the three frozen Notebook 09 pipelines with SHAP and packages reproducible artifacts for a future LLM explanation layer. It does not refit, select, or evaluate a model. Full-cohort explanations describe the fitted artifacts and are not performance estimates.


## 1. Check the environment

Confirm that the installed SHAP version is available before loading any model.


In [1]:
from importlib.metadata import version

shap_version = version("shap")
print("SHAP version:", shap_version)


SHAP version: 0.52.0


## 2. Import the analysis tools

Load only the tools needed for verification, SHAP calculation, plotting, and artifact saving.


In [2]:
from pathlib import Path
import hashlib
import io
import json
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display


## 3. Locate the frozen inputs

Use the final 26-feature table and the exact saved pipelines from Notebook 09. Save all new outputs in a separate interpretation folder.


In [3]:
working_directory = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in [working_directory, *working_directory.parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "docs/research_protocol.md").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Launch this notebook from within the project directory.")

final_root = project_root / "results/final_pipeline/06_final_fit_and_performance_summary"
full_fit_directory = final_root / "09_full_cohort_fit"
output_directory = (
    project_root
    / "results/final_pipeline/08_interpretation_and_reporting/12_shap_and_llm_handoff"
)
output_directory.mkdir(parents=True, exist_ok=True)

input_paths = {
    "predictors": final_root / "02_patient_level_aggregation/primary_1040_26_predictors.csv",
    "outcomes": final_root / "02_patient_level_aggregation/primary_1040_outcome_metadata.csv",
    "manifest": full_fit_directory / "models/model_manifest.json",
    "validation": full_fit_directory / "full_cohort_validation.csv",
}
assert all(path.is_file() for path in input_paths.values())
print("Output:", output_directory.relative_to(project_root))


Output: results/final_pipeline/08_interpretation_and_reporting/12_shap_and_llm_handoff


## 4. Load and verify the saved models

Check each model hash against the Notebook 09 manifest before explaining it.


In [4]:
def file_digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


predictors = pd.read_csv(
    input_paths["predictors"], dtype={"PATNO": "string"}, low_memory=False,
).set_index("PATNO")
outcomes = pd.read_csv(
    input_paths["outcomes"], dtype={"PATNO": "string"}, usecols=["PATNO", "falls_class"],
).set_index("PATNO")["falls_class"].astype(int)
manifest = json.loads(input_paths["manifest"].read_text())

models = {}
for target, entry in manifest["models"].items():
    model_path = full_fit_directory / "models" / entry["file"]
    assert file_digest(model_path) == entry["sha256"], f"Hash mismatch: {entry['file']}"
    with model_path.open("rb") as handle:
        models[target] = pickle.load(handle)

upstream = pd.read_csv(input_paths["validation"])
assert upstream["passed"].astype(str).str.lower().eq("true").all()
assert predictors.columns.tolist() == manifest["input_columns"]
assert predictors.index.equals(outcomes.index)
print("Verified models:", ", ".join(models))
print("Patients:", len(predictors))


Verified models: direct, stage_1, stage_2
Patients: 1040


## 5. Define clinical labels

Map the model's source groups to plain-language names used in tables and plots.


In [5]:
LABELS = {
    "Years_since_PD_diagnosis": "Years since PD diagnosis",
    "Age": "Age",
    "DXPOSINS": "Postural instability at diagnosis",
    "DXRIGID": "Rigidity at diagnosis",
    "DOPTHERST": "Dopaminergic therapy started",
    "MCATOT": "MoCA total (cognition)",
    "GROUP_FREEZING_FORM": "Freezing of gait, preceding 12 months",
    "SCAU14": "Lightheadedness on standing up (SCOPA-AUT)",
    "SCAU16": "Fainting in preceding six months (SCOPA-AUT)",
    "GDS_TOTAL": "Geriatric Depression Scale total",
    "GROUP_NEUROQOL_FORM": "Neuro-QoL Gaussian mobility score (8 items)",
    "NP1RTOT": "MDS-UPDRS Part I rater total",
    "BMI": "Body mass index",
    "NP1SLPD": "Daytime sleepiness",
    "NP1URIN": "Urinary problems",
    "NP3GAIT_COMBINED_MAX": "Gait (MDS-UPDRS Part III)",
    "NP3PSTBL_COMBINED_MAX": "Postural stability (MDS-UPDRS Part III)",
    "NHY_COMBINED_MAX": "Hoehn and Yahr stage",
    "FEATPOSHYP": "Symptomatic orthostatic hypotension documented",
    "NP3TOT_COMBINED_MAX": "MDS-UPDRS Part III motor total",
    "GROUP_PART_IV_FORM": "MDS-UPDRS Part IV total (motor complications)",
    "ANYFAMPD": "Family history of PD",
    "DXTREMOR": "Tremor at diagnosis",
    "DXBRADY": "Bradykinesia at diagnosis",
    "DOMSIDE": "Dominant side at diagnosis",
    "NP1CNST": "Constipation",
}
CLASS_LABELS = {
    "direct": ["no fall", "rare fall", "recurrent fall"],
    "stage_1": ["no fall", "any fall"],
    "stage_2": ["rare fall", "recurrent fall"],
}
assert len(LABELS) == 26
print("Clinical source groups:", len(LABELS))


Clinical source groups: 26


## 6. Recreate each model's fitted input

Apply each saved candidate transformer without refitting it. Stage 2 is explained for all patients, but it is operationally used only after Stage 1 routes a patient to “any fall.”


In [6]:
transformed = {}
for target, pipeline in models.items():
    frame = pipeline.named_steps["candidate"].transform(predictors)
    assert isinstance(frame, pd.DataFrame)
    assert frame.columns.tolist() == pipeline.named_steps["candidate"].selected_columns_
    transformed[target] = frame
    print(target, frame.shape)


direct (1040, 29)
stage_1 (1040, 29)
stage_2 (1040, 25)


## 7. Calculate SHAP values

Use `TreeExplainer` on the frozen tree estimator. Verify additivity against the estimator output: raw class scores for CatBoost and probabilities for the forest models.


In [7]:
def expected_model_output(model, frame):
    if model.__class__.__module__.startswith("catboost"):
        return np.asarray(model.predict(frame, prediction_type="RawFormulaVal"), dtype=float)
    return np.asarray(model.predict_proba(frame), dtype=float)


def explain_pipeline(pipeline, frame):
    model = pipeline.named_steps["model"]
    explanation = shap.TreeExplainer(model)(frame)
    values = np.asarray(explanation.values, dtype=float)
    base_values = np.asarray(explanation.base_values, dtype=float)
    if values.ndim == 2:
        values = values[:, :, None]
    if base_values.ndim == 1:
        base_values = np.broadcast_to(base_values, (len(frame), len(base_values)))
    expected = expected_model_output(model, frame)
    reconstructed = base_values + values.sum(axis=1)
    error = float(np.max(np.abs(reconstructed - expected)))
    return values, base_values, error


explanations = {
    target: explain_pipeline(models[target], transformed[target])
    for target in models
}
for target, (values, base_values, error) in explanations.items():
    print(target, values.shape, "max additivity error:", f"{error:.2e}")


direct (1040, 29, 3) max additivity error: 2.66e-15
stage_1 (1040, 29, 2) max additivity error: 7.27e-15
stage_2 (1040, 25, 2) max additivity error: 2.19e-15


## 8. Combine encoded columns into clinical groups

Sum one-hot and missing-indicator SHAP contributions back to their clinical source group. This preserves additivity while making the handoff understandable.


In [8]:
def grouped_explanation(target):
    transformer = models[target].named_steps["candidate"]
    columns = transformed[target].columns.tolist()
    groups = list(dict.fromkeys(transformer.group_by_column_[column] for column in columns))
    values = explanations[target][0]
    grouped = np.stack([
        values[:, [i for i, column in enumerate(columns)
                   if transformer.group_by_column_[column] == group], :].sum(axis=1)
        for group in groups
    ], axis=1)
    return groups, grouped


grouped = {target: grouped_explanation(target) for target in models}
for target, (groups, values) in grouped.items():
    assert np.allclose(values.sum(axis=1), explanations[target][0].sum(axis=1))
    print(target, len(groups), "clinical groups")


direct 20 clinical groups
stage_1 20 clinical groups
stage_2 13 clinical groups


## 9. Summarize global SHAP magnitude

Average absolute grouped SHAP values across patients for each model output class. Magnitude describes reliance; it does not establish causation or direction by itself.


In [9]:
summary_rows = []
for target, (groups, values) in grouped.items():
    for class_index, class_label in enumerate(CLASS_LABELS[target]):
        for group_index, group in enumerate(groups):
            class_values = values[:, group_index, class_index]
            summary_rows.append({
                "target": target,
                "class_index": class_index,
                "class_label": class_label,
                "group": group,
                "descriptive_name": LABELS[group],
                "mean_absolute_shap": float(np.mean(np.abs(class_values))),
                "mean_shap": float(np.mean(class_values)),
            })

shap_summary = pd.DataFrame(summary_rows).sort_values(
    ["target", "class_index", "mean_absolute_shap"], ascending=[True, True, False],
)
display(shap_summary.groupby(["target", "class_label"], sort=False).head(5).round(4))


,target,class_index,class_label,group,descriptive_name,mean_absolute_shap,mean_shap
7,direct,0,no fall,GROUP_NEUROQOL_FORM,Neuro-QoL Gaussian mobility score (8 items),0.2019,0.0692
0,direct,0,no fall,Years_since_PD_diagnosis,Years since PD diagnosis,0.1547,0.0530
15,direct,0,no fall,GROUP_PART_IV_FORM,MDS-UPDRS Part IV total (motor complications),0.1223,0.0445
3,direct,0,no fall,GROUP_FREEZING_FORM,"Freezing of gait, preceding 12 months",0.1199,0.0576
1,direct,0,no fall,Age,Age,0.1005,0.0263
36,direct,1,rare fall,NP1CNST,Constipation,0.1102,-0.0226
22,direct,1,rare fall,MCATOT,MoCA total (cognition),0.0854,-0.0110
23,direct,1,rare fall,GROUP_FREEZING_FORM,"Freezing of gait, preceding 12 months",0.0854,0.0332
21,direct,1,rare fall,Age,Age,0.0738,-0.0124
39,direct,1,rare fall,ANYFAMPD,Family history of PD,0.0501,0.0044


## 10. Plot the direct and two-stage components

Keep one all-class magnitude overview for the direct model, then show full beeswarm plots for every direct class and the positive class of each two-stage component. Every plot names the fitted model and includes every clinical source group retained by that model.


In [10]:
GROUP_SOURCE = {
    "GROUP_FREEZING_FORM": "FRZGT12M",
    "GROUP_NEUROQOL_FORM": "NQ_GAUSSIAN_REVISION",
    "GROUP_PART_IV_FORM": "NP4TOT",
}
CATEGORY_CODES = {
    "No": 0.0,
    "Yes": 1.0,
    "Uncertain": 0.5,
    "Unknown": 2.0,
    "Left": 0.0,
    "Right": 1.0,
    "Symmetric": 2.0,
}
MODEL_TITLES = {
    "direct": "Direct balanced CatBoost",
    "stage_1": "Two-stage Stage 1 balanced Extra Trees",
    "stage_2": "Two-stage Stage 2 random forest",
}


def group_display_frame(target):
    groups = grouped[target][0]
    display_values = {}
    for group in groups:
        source = GROUP_SOURCE.get(group, group)
        values = predictors[source]
        if not pd.api.types.is_numeric_dtype(values):
            values = values.map(CATEGORY_CODES)
        display_values[LABELS[group]] = pd.to_numeric(values, errors="coerce")
    return pd.DataFrame(display_values, index=predictors.index)


group_display = {target: group_display_frame(target) for target in models}


def save_all_class_overview():
    subset = (
        shap_summary.loc[shap_summary["target"].eq("direct")]
        .groupby("descriptive_name", as_index=False)["mean_absolute_shap"].mean()
        .sort_values("mean_absolute_shap")
    )
    figure, axis = plt.subplots(figsize=(9, 8))
    axis.barh(subset["descriptive_name"], subset["mean_absolute_shap"], color="#31688e")
    axis.set_xlabel("Mean absolute SHAP value across three class outputs")
    axis.set_title("Direct balanced CatBoost — all 20 retained clinical features")
    figure.tight_layout()
    figure.savefig(
        output_directory / "shap_summary_direct_all_classes.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.close(figure)


def save_class_beeswarm(target, class_label, filename):
    class_index = CLASS_LABELS[target].index(class_label)
    groups, grouped_values = grouped[target]
    display_frame = group_display[target]
    assert display_frame.shape[1] == len(groups)

    shap.summary_plot(
        grouped_values[:, :, class_index],
        features=display_frame,
        feature_names=display_frame.columns.tolist(),
        max_display=len(groups),
        show=False,
        plot_size=(10, max(6, 0.38 * len(groups) + 1.8)),
    )
    figure = plt.gcf()
    plt.title(
        f"{MODEL_TITLES[target]} — SHAP toward {class_label}",
        pad=12,
    )
    figure.text(
        0.5,
        0.005,
        "Each point is one patient. Color is the raw feature value; categorical colors are codes, not an order.",
        ha="center",
        fontsize=8,
    )
    figure.savefig(output_directory / filename, dpi=180, bbox_inches="tight")
    plt.close(figure)


save_all_class_overview()
beeswarm_specs = [
    ("direct", "no fall", "shap_summary_direct_no_fall.png"),
    ("direct", "rare fall", "shap_summary_direct_rare_fall.png"),
    ("direct", "recurrent fall", "shap_summary_direct_recurrent_fall.png"),
    ("stage_1", "any fall", "shap_summary_stage1.png"),
    ("stage_2", "recurrent fall", "shap_summary_stage2.png"),
]
for specification in beeswarm_specs:
    save_class_beeswarm(*specification)

plot_specs = [
    ("direct", None, None, "shap_summary_direct_all_classes.png"),
    *[(target, class_label, None, filename) for target, class_label, filename in beeswarm_specs],
]
print("Plots created: 1 all-class overview and 5 full-feature beeswarms")


Plots created: 1 all-class overview and 5 full-feature beeswarms


## 11. Build the transformed-feature map

Connect every encoded model column to its clinical source group, plain-language label, and the models that retain it.


In [11]:
all_columns = list(dict.fromkeys(
    column for target in models for column in transformed[target].columns
))
feature_rows = []
for column in all_columns:
    source_group = next(
        models[target].named_steps["candidate"].group_by_column_[column]
        for target in models if column in transformed[target].columns
    )
    if column.endswith("_MISSING") or "_Missing" in column:
        column_type = "missingness indicator"
    elif any(column.startswith(f"{name}_") for name in [
        "DXPOSINS", "DXRIGID", "DOPTHERST", "FEATPOSHYP",
        "ANYFAMPD", "DXTREMOR", "DXBRADY", "DOMSIDE",
    ]):
        column_type = "one-hot category"
    else:
        column_type = "clinical value"
    feature_rows.append({
        "model_column": column,
        "source_group": source_group,
        "descriptive_name": LABELS[source_group],
        "column_type": column_type,
        "used_by_direct": column in transformed["direct"].columns,
        "used_by_stage_1": column in transformed["stage_1"].columns,
        "used_by_stage_2": column in transformed["stage_2"].columns,
    })
feature_map = pd.DataFrame(feature_rows)
display(feature_map.head(10))


,model_column,source_group,descriptive_name,column_type,used_by_direct,used_by_stage_1,used_by_stage_2
0,Years_since_PD_diagnosis,Years_since_PD_diagnosis,Years since PD diagnosis,clinical value,True,True,True
1,Age,Age,Age,clinical value,True,True,False
2,MCATOT,MCATOT,MoCA total (cognition),clinical value,True,True,False
3,FRZGT12M,GROUP_FREEZING_FORM,"Freezing of gait, preceding 12 months",clinical value,True,True,True
4,SCAU14,SCAU14,Lightheadedness on standing up (SCOPA-AUT),clinical value,True,True,True
5,SCAU16,SCAU16,Fainting in preceding six months (SCOPA-AUT),clinical value,True,True,False
6,GDS_TOTAL,GDS_TOTAL,Geriatric Depression Scale total,clinical value,True,True,False
7,NQ_GAUSSIAN_REVISION,GROUP_NEUROQOL_FORM,Neuro-QoL Gaussian mobility score (8 items),clinical value,True,True,True
8,NP1RTOT,NP1RTOT,MDS-UPDRS Part I rater total,clinical value,True,True,False
9,NP1SLPD,NP1SLPD,Daytime sleepiness,clinical value,True,True,False


## 12. Build the patient alignment table

Record row order and model outputs so every saved SHAP row can be matched safely. This file contains PPMI participant identifiers and must follow PPMI data-use restrictions.


In [12]:
direct_probabilities = np.asarray(models["direct"].predict_proba(predictors), dtype=float)
stage_1_probabilities = np.asarray(models["stage_1"].predict_proba(predictors), dtype=float)
stage_2_probabilities = np.asarray(models["stage_2"].predict_proba(predictors), dtype=float)

direct_class = np.asarray(models["direct"].predict(predictors), dtype=int).reshape(-1)
stage_1_class = np.asarray(models["stage_1"].predict(predictors), dtype=int).reshape(-1)
stage_2_class = np.asarray(models["stage_2"].predict(predictors), dtype=int).reshape(-1)
two_stage_class = np.where(stage_1_class == 0, 0, stage_2_class + 1)
two_stage_probabilities = np.column_stack([
    stage_1_probabilities[:, 0],
    stage_1_probabilities[:, 1] * stage_2_probabilities[:, 0],
    stage_1_probabilities[:, 1] * stage_2_probabilities[:, 1],
])

patient_index = pd.DataFrame({
    "row_index": np.arange(len(predictors)),
    "PATNO": predictors.index,
    "observed_falls_class": outcomes.to_numpy(),
    "direct_predicted_class": direct_class,
    "direct_probability_no_fall": direct_probabilities[:, 0],
    "direct_probability_rare_fall": direct_probabilities[:, 1],
    "direct_probability_recurrent_fall": direct_probabilities[:, 2],
    "stage_1_predicted_any_fall": stage_1_class,
    "stage_1_probability_any_fall": stage_1_probabilities[:, 1],
    "stage_2_predicted_recurrent_fall": stage_2_class,
    "stage_2_probability_recurrent_fall": stage_2_probabilities[:, 1],
    "two_stage_predicted_class": two_stage_class,
    "two_stage_probability_no_fall": two_stage_probabilities[:, 0],
    "two_stage_probability_rare_fall": two_stage_probabilities[:, 1],
    "two_stage_probability_recurrent_fall": two_stage_probabilities[:, 2],
})
display(patient_index.head(3))


,row_index,PATNO,observed_falls_class,direct_predicted_class,direct_probability_no_fall,direct_probability_rare_fall,direct_probability_recurrent_fall,stage_1_predicted_any_fall,stage_1_probability_any_fall,stage_2_predicted_recurrent_fall,stage_2_probability_recurrent_fall,two_stage_predicted_class,two_stage_probability_no_fall,two_stage_probability_rare_fall,two_stage_probability_recurrent_fall
0,0,100001,0,0,0.488474,0.426998,0.084528,0,0.302558,0,0.283333,0,0.697442,0.216833,0.085725
1,1,100002,0,0,0.648541,0.266539,0.084920,0,0.270760,0,0.150000,0,0.729240,0.230146,0.040614
2,2,100005,1,0,0.658104,0.255122,0.086773,0,0.402999,0,0.066667,0,0.597001,0.376132,0.026867


## 13. Define safe artifact writers

Allow an exact or numerically equivalent rerun, but refuse to replace a meaningfully different artifact.


In [13]:
def save_csv_new_or_equivalent(frame, path):
    if path.is_file():
        existing = pd.read_csv(path, dtype={"PATNO": "string"} if "PATNO" in frame else None)
        current = pd.read_csv(
            io.StringIO(frame.to_csv(index=False)),
            dtype={"PATNO": "string"} if "PATNO" in frame else None,
        )
        pd.testing.assert_frame_equal(current, existing, check_exact=False, rtol=1e-10)
        return "already equivalent"
    frame.to_csv(path, index=False)
    return "created"


def save_npz_new_or_equivalent(path, **arrays):
    if path.is_file():
        with np.load(path) as existing:
            assert set(existing.files) == set(arrays)
            for key, current in arrays.items():
                saved = existing[key]
                if np.issubdtype(np.asarray(current).dtype, np.number):
                    np.testing.assert_allclose(saved, current, rtol=1e-10, atol=1e-12)
                else:
                    np.testing.assert_array_equal(saved, current)
        return "already equivalent"
    np.savez_compressed(path, **arrays)
    return "created"


def save_text_new_or_identical(text, path):
    if path.is_file():
        assert path.read_text() == text, f"Existing artifact differs: {path.name}"
        return "already identical"
    path.write_text(text)
    return "created"


## 14. Validate the explanation package

Require model-hash integrity, SHAP additivity, complete patient alignment, grouped additivity, and all requested plots.


In [14]:
validation = pd.DataFrame([
    {
        "check": "all model hashes match Notebook 09",
        "passed": all(
            file_digest(full_fit_directory / "models" / entry["file"]) == entry["sha256"]
            for entry in manifest["models"].values()
        ),
    },
    {
        "check": "SHAP covers direct, Stage 1, and Stage 2",
        "passed": set(explanations) == {"direct", "stage_1", "stage_2"},
    },
    {
        "check": "all SHAP values are finite",
        "passed": all(np.isfinite(values).all() for values, _, _ in explanations.values()),
    },
    {
        "check": "SHAP additivity error is below 1e-6",
        "passed": all(error < 1e-6 for _, _, error in explanations.values()),
    },
    {
        "check": "grouped SHAP preserves additivity",
        "passed": all(
            np.allclose(grouped[target][1].sum(axis=1), explanations[target][0].sum(axis=1))
            for target in models
        ),
    },
    {
        "check": "patient alignment covers all 1,040 patients",
        "passed": len(patient_index) == 1040 and patient_index["PATNO"].is_unique,
    },
    {
        "check": "two-stage probabilities sum to one",
        "passed": bool(np.allclose(two_stage_probabilities.sum(axis=1), 1.0)),
    },
    {
        "check": "six requested SHAP plots exist",
        "passed": all((output_directory / item[3]).is_file() for item in plot_specs),
    },
])
display(validation)
assert validation["passed"].all()


,check,passed
0,all model hashes match Notebook 09,True
1,"SHAP covers direct, Stage 1, and Stage 2",True
2,all SHAP values are finite,True
3,SHAP additivity error is below 1e-6,True
4,grouped SHAP preserves additivity,True
5,"patient alignment covers all 1,040 patients",True
6,two-stage probabilities sum to one,True
7,six requested SHAP plots exist,True


## 15. Save the SHAP and LLM handoff artifacts

Store encoded and grouped SHAP arrays, model inputs, row alignment, feature mapping, summaries, and a machine-readable manifest.


In [15]:
save_rows = []
for target in models:
    groups, grouped_values = grouped[target]
    values, base_values, error = explanations[target]
    path = output_directory / f"{target}_shap_artifact.npz"
    status = save_npz_new_or_equivalent(
        path,
        shap_values=values,
        base_values=base_values,
        model_input_values=transformed[target].to_numpy(dtype=float),
        model_feature_names=np.asarray(transformed[target].columns, dtype=str),
        grouped_shap_values=grouped_values,
        group_names=np.asarray(groups, dtype=str),
        class_names=np.asarray(CLASS_LABELS[target], dtype=str),
        patient_ids=predictors.index.to_numpy(dtype=str),
    )
    save_rows.append({"artifact": path.name, "status": status})

for filename, frame in {
    "shap_global_summary.csv": shap_summary,
    "feature_map.csv": feature_map,
    "patient_explanation_index.csv": patient_index,
    "shap_validation.csv": validation,
}.items():
    save_rows.append({
        "artifact": filename,
        "status": save_csv_new_or_equivalent(frame, output_directory / filename),
    })

handoff_manifest = {
    "run_version": "final-notebook-12-shap-llm-handoff-v2-full-beeswarms",
    "source_model_manifest_sha256": file_digest(input_paths["manifest"]),
    "shap_version": shap_version,
    "patients": len(predictors),
    "models": {
        target: {
            "model_file": manifest["models"][target]["file"],
            "model_sha256": manifest["models"][target]["sha256"],
            "classes": CLASS_LABELS[target],
            "model_columns": len(transformed[target].columns),
            "clinical_groups": len(grouped[target][0]),
            "shap_shape": list(explanations[target][0].shape),
            "output_scale": "raw class score" if target == "direct" else "class probability",
            "max_additivity_error": float(f"{explanations[target][2]:.3e}"),
        }
        for target in models
    },
    "plots": {
        "direct_all_classes": "all 20 retained clinical groups; mean absolute SHAP overview",
        "class_specific": "full beeswarm for every direct class, Stage 1 any fall, and Stage 2 recurrent fall",
    },
    "two_stage_interpretation": (
        "Stage 1 and Stage 2 are separate hard-routed components. Do not add their SHAP values. "
        "Stage 2 applies only when Stage 1 predicts any fall."
    ),
    "performance_boundary": (
        "These full-cohort explanations are not performance estimates; use Notebook 07 outer evaluation."
    ),
    "governance": (
        "NPZ files, patient_explanation_index.csv, and patient_raw_inputs.csv contain "
        "patient-level PPMI-derived data. Public sharing was explicitly approved for this project export."
    ),
}
manifest_text = json.dumps(handoff_manifest, indent=2) + "\n"
manifest_output_path = output_directory / "shap_handoff_manifest.json"
manifest_existed = manifest_output_path.is_file()
manifest_status = "already identical"
if not manifest_existed or manifest_output_path.read_text() != manifest_text:
    manifest_output_path.write_text(manifest_text)
    manifest_status = "updated to v2" if manifest_existed else "created"
save_rows.append({
    "artifact": "shap_handoff_manifest.json",
    "status": manifest_status,
})

display(pd.DataFrame(save_rows))


,artifact,status
0,direct_shap_artifact.npz,already equivalent
1,stage_1_shap_artifact.npz,already equivalent
2,stage_2_shap_artifact.npz,already equivalent
3,shap_global_summary.csv,already equivalent
4,feature_map.csv,already equivalent
5,patient_explanation_index.csv,already equivalent
6,shap_validation.csv,already equivalent
7,shap_handoff_manifest.json,updated to v2


## Interpretation boundary

SHAP describes how the frozen fitted models use their inputs; it does not show causation, treatment effect, or unbiased predictive performance. Stage 1 and Stage 2 explanations remain separate because the hard routing step is not an additive model. Patient-level files remain governed PPMI-derived data and should not be placed in a public repository without confirming permission.
